# Tutorial 2 — More Leptons

A guided closure test for [Machine Learning Neutrino-Nucleus Cross Sections II: More Leptons](https://arxiv.org/abs/2610.30338). Start with synthetic near-detector (ND) distributions, learn cross sections, generate far-detector (FD) events, and compare three-parameter oscillation fits with the true and learned cross sections.

Run `python -m pip install -e '.[notebook]'` from the repository root, open this notebook there, and select **Restart Kernel and Run All**. The PDF grids and flux tables are included; LHAPDF and a GPU are not required. The final optional section introduces flux systematics.

The default **quick** profile teaches the entire nominal workflow with reduced training and FD statistics. **Full** increases the numerical budget; **smoke** only checks that the workflow executes. None of these settings alone certifies the published contours: check training and integration convergence, and use an ensemble of systematic draws for the paper's total uncertainties.

## Setup: local CPU or Google Colab

Locally, install with `python -m pip install -e '.[notebook]'` from the cloned repository, then open this notebook in Jupyter. The setup cell finds the repository root automatically.

In Colab, choose a Python runtime with **no accelerator** and run this setup cell first. It clones `main`, installs the package, and makes its input files available. If this repository is private, select `COLAB_SOURCE="upload"` and upload its repository ZIP when prompted. Instructions are in the [README](https://github.com/karlaTame/ML_nuxsec_school/blob/main/README.md). Start with `PROFILE="smoke"` in the next configuration cell for a short execution check; use `"quick"` for the tutorial. CPU is sufficient for both.

Runtime outputs stay under `outputs/`; source notebooks ship without outputs or saved machine-specific kernels. Colab runtime files are temporary, so download any results you want to retain before disconnecting.


In [ ]:
# Local: install .[notebook] first. Colab: this cell prepares the checkout.
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

REPO_URL = "https://github.com/karlaTame/ML_nuxsec_school.git"
REPO_BRANCH = "main"  # change this when using a fork or another branch
COLAB_SOURCE = "github"  # github | upload (ZIP of this repository, for private repos)


def prepare_colab(repo_url, branch, target, source="github", archive=None):
    target = Path(target).resolve()
    if not (target / "pyproject.toml").is_file():
        if target.exists():
            raise RuntimeError(f"{target} exists but is not this repository; choose another folder.")
        if source == "upload":
            if archive is None:
                from google.colab import files
                uploaded = files.upload()  # choose the repository ZIP, not just the notebook
                if len(uploaded) != 1:
                    raise ValueError("Upload exactly one ZIP of the repository.")
                archive = next(iter(uploaded))
            with tempfile.TemporaryDirectory() as staging:
                staging = Path(staging)
                with zipfile.ZipFile(archive) as bundle:
                    for member in bundle.infolist():
                        if not (staging / member.filename).resolve().is_relative_to(staging.resolve()):
                            raise ValueError("ZIP contains a path outside its extraction folder.")
                    bundle.extractall(staging)
                roots = [p.parent for p in staging.rglob("pyproject.toml")
                         if (p.parent / "src" / "ml4nu").is_dir()]
                if len(roots) != 1:
                    raise ValueError("ZIP must contain exactly one complete tutorial repository.")
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(roots[0]), str(target))
        elif source == "github":
            try:
                subprocess.run(["git", "clone", "--depth", "1", "--branch", branch,
                                repo_url, str(target)], check=True)
            except subprocess.CalledProcessError as error:
                raise RuntimeError("Clone failed. For a private repository, set COLAB_SOURCE='upload' "
                                   "and upload the ZIP of this branch. Check repository access and branch name.") from error
        else:
            raise ValueError("COLAB_SOURCE must be 'github' or 'upload'.")
    if not (target / "src" / "ml4nu").is_dir():
        raise RuntimeError("The selected folder is not a complete tutorial checkout.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e",
                    str(target) + "[colab]"], check=True)
    # pip's editable .pth is processed only when Python starts; this kernel is already running.
    sys.path.insert(0, str(target / "src"))
    importlib.invalidate_caches()
    os.chdir(target)
    return target


try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = prepare_colab(REPO_URL, REPO_BRANCH, "/content/ML_nuxsec_school", COLAB_SOURCE)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "pyproject.toml").is_file() and (p / "src" / "ml4nu").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Open this notebook from your cloned ML_nuxsec_school repository.")
    os.chdir(ROOT)
print("Repository:", ROOT)


## 1. Choose a run size

All outputs go under `outputs/more_leptons/<profile>/`. Seeds control network initialization, event sampling, and bootstrap resampling separately. Re-running a profile regenerates its artifacts. CPU is the default so the notebook also runs on a laptop; set `DEVICE="cuda"` for an available GPU. Do not select a GPU profile that exceeds its memory.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform

DEVICE = "cpu"  # cpu | cuda; restart the kernel after changing this

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from ml4nu.pdg import pid
from ml4nu.utils import grab, make_marginal_rate
from ml4nu.pipeline import rates_common as rc
from ml4nu.pipeline.train_near_detector import train_near_detector_varpro
from ml4nu.pipeline.generate_fd_samples import generate_fd_samples
from ml4nu.pipeline import fd_optimize_torch as ft

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Start Jupyter in the ml4nu repository root.")
PROFILE = os.environ.get("ML4NU_PROFILE", "quick")  # quick | full | smoke
PROFILES = {
    "smoke": dict(steps=40, enu_nd=32, v_nd=8, events=200,
                  boots=4, enu_fd=48, v_fd=24, starts=2, max_iter=50),
    "quick": dict(steps=2000, enu_nd=64, v_nd=16, events=1000,
                  boots=32, enu_fd=128, v_fd=64, starts=4, max_iter=300),
    "full": dict(steps=100000, enu_nd=64, v_nd=16, events=6200,
                 boots=4096, enu_fd=512, v_fd=256, starts=4, max_iter=2000),
}
if PROFILE not in PROFILES:
    raise ValueError(f"Unknown profile: {PROFILE}")
cfg = PROFILES[PROFILE]
seeds = dict(training=17, samples=11, bootstrap=0)
torch.set_default_device("cpu")
torch.set_default_dtype(torch.float64)
torch.set_num_threads(2)
torch.set_grad_enabled(False)
rc.device = torch.device(DEVICE)
from ml4nu import sampling
sampling.device = rc.device
OUT = ROOT / "outputs" / "more_leptons" / PROFILE
OUT.mkdir(parents=True, exist_ok=True)
print(f"Profile: {PROFILE}; device: {rc.device}")
display(pd.Series(cfg, name="settings").to_frame())

## 2. Understand the channels and inputs

Each horn mode contains neutrinos and antineutrinos. Separate five-output networks describe their structure functions; electron and muon cross sections share each network, with the lepton mass entering the known kinematic factors.

| Beam mode | Components | ND training observations | FD likelihood observations |
|---|---|---|---|
| FHC (`fwd`) | `mu`, `mubar`, `e`, `ebar` | $\mu^-$, $\mu^+$, $e^-+e^+$ | $\mu^-+\mu^+$, $e^-+e^+$ |
| RHC (`bwd`) | same four | same three | same two |

There are eight component rates, six ND training histograms, and four charge-inclusive FD samples. In particular, electron and positron components are summed **before** comparing the model with ND data.

The supplied grids define the synthetic truth, including its low-$Q^2$ prescription; they are not training labels for the learned structure functions. The four GLoBES tables give independent unoscillated ND/FD flux shapes. See [data/README.md](data/README.md) for their provenance and [the code map](docs/tutorials.md) for the implementation.

In [ ]:
required = [Path(rc.PDF_FN), Path(rc.PDFB_FN),
            Path(rc.FLUX_FILE_FWD), Path(rc.FLUX_FILE_BWD),
            Path(rc.FLUX_FILE_FD_FWD), Path(rc.FLUX_FILE_FD_BWD)]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Missing repository inputs: {missing}")
setup = rc.build_setup(N_Enu_ND=cfg["enu_nd"], N_v_ND=cfg["v_nd"])
fluxes, targets = rc.build_training_inputs(setup)
phi = dict(zip(rc.CHANNELS, fluxes))
P = dict(zip(rc.CHANNELS, targets))
print("ND energy range:", float(setup.kin_grid_mu.all_enu.min()), rc.ENUMAX, "GeV")
print("FD analysis window:", rc.FD_EWINDOW, "GeV")
display(pd.DataFrame([setup.osc_params], index=["injected oscillations"]))

In [ ]:
energies = np.linspace(0.2, rc.ENUMAX, 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
flavors = [(pid.numu, r"$\nu_\mu$"), (pid.numubar, r"$\bar\nu_\mu$"),
           (pid.nue, r"$\nu_e$"), (pid.nuebar, r"$\bar\nu_e$")]
for ax, flux, title in zip(axes, [setup.nd_flux_fwd, setup.nd_flux_bwd], ["FHC", "RHC"]):
    for flavor, label in flavors:
        ax.plot(energies, grab(flux(flavor, energies)), label=label)
    ax.set(title=title, xlabel=r"$E_\nu$ [GeV]", yscale="log")
axes[0].set_ylabel("ND flux [table units]")
axes[1].legend()
fig.tight_layout()
plt.show()

## 3. Learn cross sections from the ND

On the kinematic grid, `make_marginal_rate` integrates the cross section over the unobserved neutrino energy against the channel flux. It preserves the rate normalization. The coordinates are $v_1=\sqrt{E_\ell}$ and $v_2=\sqrt{E_\ell}\sin(\theta/2)$.

`train_near_detector_varpro` fits the ND histograms jointly. At each step it solves the linear output coefficients by least squares, then updates the network features with Adam. The checkpoint contains the best loss weights for both networks. Truth cross sections are used only for diagnostics; the loss uses the ND histograms. A short run may fit these histograms well while still extrapolating poorly to FD fluxes.

In [ ]:
model_path = train_near_detector_varpro(
    setup, fluxes, targets,
    N_steps=cfg["steps"], seed=seeds["training"], lr=1e-3, n_sf=5,
    out_dir=str(OUT / "models"), tag=PROFILE,
)
# Inspect raw predictions before applying inference-side positivity.
model, model_bar, raw_calc, raw_calc_bar = rc.load_model(model_path, setup)
ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
print("Best ND loss:", ckpt["meta"]["best_loss"],
      "at step", ckpt["meta"]["best_loss_step"])

In [ ]:
pred = {}
negative_fraction = {}
leptons = {"mu": pid.mu, "mubar": pid.mubar, "e": pid.e, "ebar": pid.ebar}
for base, flavor in leptons.items():
    kg = setup.kin_grid_mu if base in ("mu", "mubar") else setup.kin_grid_e
    calc = raw_calc_bar if base in ("mubar", "ebar") else raw_calc
    ds = calc(kg.grid_enu, kg.grid1, kg.grid2,
              flavor=flavor, kin_mapping=setup.kin_mapping)
    negative_fraction[base] = float((ds < 0).double().mean())
    for mode in ("fwd", "bwd"):
        ch = f"{base}_{mode}"
        pred[ch] = make_marginal_rate(ds, phi[ch], kg.denu, kg.dV2)

fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharex=True)
residuals = []
for row, mode in enumerate(("fwd", "bwd")):
    for col, bases in enumerate((("mu",), ("mubar",), ("e", "ebar"))):
        data = sum(P[f"{b}_{mode}"] for b in bases)
        learned = sum(pred[f"{b}_{mode}"] for b in bases)
        kg = setup.kin_grid_mu if col < 2 else setup.kin_grid_e
        ax = axes[row, col]
        ax.plot(grab(kg.all_var1), grab(data.sum(1) * kg.dvar2), "k--", label="true")
        ax.plot(grab(kg.all_var1), grab(learned.sum(1) * kg.dvar2), label="learned")
        title = f"{'FHC' if mode == 'fwd' else 'RHC'}: {' + '.join(bases)}"
        ax.set_title(title)
        residuals.append(dict(observation=title,
                              relative_L2=float(torch.linalg.vector_norm(learned-data)
                                                / torch.linalg.vector_norm(data))))
        if row == 1:
            ax.set_xlabel(r"$v_1$ [GeV$^{1/2}$]")
axes[0, 0].legend()
axes[0, 0].set_ylabel("ND marginal rate [arbitrary units]")
axes[1, 0].set_ylabel("ND marginal rate [arbitrary units]")
fig.tight_layout()
plt.show()
display(pd.DataFrame(residuals))
print("Raw negative cross-section grid fractions:", negative_fraction)

## 4. Generate FD data with the known cross section

Oscillations change the flux while the cross section stays fixed. A single exposure converts rates into event counts, anchored to the FHC $\mu^-$ count in the configuration. The same exposure fixes all other channels. The exposure is saved alongside the samples and reused in the likelihood.

Accept/reject sampling writes eight arrays of `(E_nu, E_lepton, cos_theta)`. True neutrino energy is available only because this is simulated data: the oscillation fit uses the observed lepton coordinates and integrates over neutrino energy. Both signs of charge are then combined into four FD samples.

In [ ]:
sample_dir = OUT / "samples"
generate_fd_samples(setup, seed=seeds["samples"],
                    sample_dir=str(sample_dir), num_mu_fwd=cfg["events"])
counts = {ch: len(np.load(sample_dir / f"samples_FD_{ch}.npy"))
          for ch in rc.CHANNELS}
display(pd.Series(counts, name="generated events").to_frame())
print("Saved exposure:", rc.load_exposure(str(sample_dir)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for mode, style in (("fwd", "-"), ("bwd", "--")):
    for bases in (("mu", "mubar"), ("e", "ebar")):
        events = np.concatenate([np.load(sample_dir / f"samples_FD_{b}_{mode}.npy")
                                 for b in bases])
        label = f"{'FHC' if mode == 'fwd' else 'RHC'} {'mu-like' if bases[0]=='mu' else 'e-like'}"
        axes[0].hist(events[:, 1], bins=30, histtype="step", linestyle=style, label=label)
        axes[1].hist(events[:, 2], bins=30, histtype="step", linestyle=style)
axes[0].set(xlabel=r"$E_\ell$ [GeV]", ylabel="events / bin")
axes[1].set(xlabel=r"$\cos\theta$", ylabel="events / bin")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

## 5. Prepare the joint likelihood

The FD calculation precomputes differential cross sections at each observed lepton coordinate and their matching total integrals. It combines the neutrino and antineutrino intensities **before** taking a logarithm. The extended Poisson likelihood retains both event shapes and absolute counts. Only oscillation parameters vary in the FD fit; network weights are frozen.

For this demonstration, negative learned cross sections are clamped to zero at inference and a small positive density floor stabilizes the logarithm. The raw negative fractions above and the floor counts below make this approximation visible. Frequent clipping/flooring is a reason to improve training before interpreting results.

In [ ]:
_, _, calc, calc_bar = rc.load_model(model_path, setup, force_ds_pos=True)
fd = rc.load_fd_samples(setup, calc, calc_bar,
                        N_Enu_FD=cfg["enu_fd"], N_v_FD=cfg["v_fd"],
                        sample_dir=str(sample_dir))
ft.LIKELIHOOD = "extended"
ft.EXPOSURE = rc.load_exposure(str(sample_dir))  # use the exposure that made these data
print("Four likelihood samples:")
display(pd.Series({ch: fd.n_samps[ch]
                   for ch in ("mutot_fwd", "etot_fwd", "mutot_bwd", "etot_bwd")},
                  name="events").to_frame())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for ax, base in zip(axes.flat, ("mu", "mubar", "e", "ebar")):
    ax.plot(grab(fd.all_Enu_mu), grab(fd.I[base]), "k--", label="true")
    ax.plot(grab(fd.all_Enu_mu), grab(fd.I_model[base]), label="learned")
    ax.set(title=base, ylabel=r"$\sigma(E_\nu)$ [arbitrary units]")
for ax in axes[-1]:
    ax.set_xlabel(r"$E_\nu$ [GeV]")
axes[0, 0].legend()
fig.tight_layout()
plt.show()

## 6. Fit oscillation parameters and bootstrap

The fit varies $(\sin^2\theta_{23},\Delta m^2_{31},\delta_{CP})$. Multiple starting points explore both octants. Each bootstrap replicate allows the sample counts to fluctuate, as required for a rate-sensitive likelihood. True and learned cross-section fits use the same resampled data.

`minima_c` stores truth fits and `minima_c_` learned fits. Their columns are **sin-squared theta23**, **Delta m-squared31 in eV-squared**, and **deltaCP in radians**. Internally the optimizer uses a bounded transformation for the angle and a positive transformation for the mass splitting.

In [ ]:
fit_path = ft.run_fd_fit(
    fd, rc.device, model_path=str(model_path), seed=seeds["bootstrap"],
    N_boots=cfg["boots"], N_Enu_FD=cfg["enu_fd"], N_v_FD=cfg["v_fd"],
    starts=cfg["starts"], max_iter=cfg["max_iter"],
    polish_steps=0 if PROFILE == "smoke" else 10, max_batch=32,
    log_floor=1e-10, tag=PROFILE, out_dir=str(OUT / "results"),
    extra_meta=dict(profile=PROFILE, likelihood=ft.LIKELIHOOD,
                    exposure=ft.EXPOSURE, configuration=cfg, seeds=seeds),
)
with np.load(fit_path, allow_pickle=True) as saved:
    fits = {"true": saved["minima_c"], "learned": saved["minima_c_"]}
    floor_counts = saved["nflr_c_"]
    diagnostics = {name: saved[name] for name in
                   ("gmax_c", "gmax_c_", "found_c", "found_c_")}
for name, values in fits.items():
    assert values.shape == (cfg["boots"], 3) and np.isfinite(values).all(), name
print("Bootstraps with floored events:", int((floor_counts > 0).sum()), "/", cfg["boots"])
print("Largest reported gradient (truth / learned):",
      float(np.max(diagnostics["gmax_c"])), float(np.max(diagnostics["gmax_c_"])))

In [ ]:
# Center the periodic CP coordinate on the injected phase for readable plots.
injected = np.array([rc.s23_true**2, rc.d31_true, rc.dcp_true])
def plot_coordinates(values):
    x = values.copy()
    x[:, 1] *= 1e3
    x[:, 2] = np.rad2deg(injected[2] + (x[:, 2] - injected[2] + np.pi) % (2*np.pi) - np.pi)
    return x

reference = plot_coordinates(injected[None, :])[0]
labels = [r"$\sin^2\theta_{23}$", r"$\Delta m^2_{31}$ [$10^{-3}$ eV$^2$]",
          r"$\delta_{CP}$ [degrees, centered on injection]"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
summary = []
for name, values in fits.items():
    shown = plot_coordinates(values)
    for j, ax in enumerate(axes):
        ax.hist(shown[:, j], bins=min(20, max(4, cfg["boots"]//2)),
                histtype="step", label=name)
        q16, median, q84 = np.quantile(shown[:, j], [0.16, 0.5, 0.84])
        summary.append(dict(cross_section=name, parameter=["sin2_theta23", "dm31_x1000", "deltaCP_deg"][j],
                            q16=q16, median=median, q84=q84, injected=reference[j]))
for j, ax in enumerate(axes):
    ax.axvline(reference[j], color="black", linestyle="--", label="injected")
    ax.set(xlabel=labels[j], ylabel="bootstrap fits / bin")
axes[0].legend()
fig.tight_layout()
plt.show()
display(pd.DataFrame(summary).round(5))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (i, j) in zip(axes, ((0, 1), (0, 2), (1, 2))):
    for name, values in fits.items():
        shown = plot_coordinates(values)
        ax.scatter(shown[:, i], shown[:, j], s=12, alpha=0.5, label=name)
    ax.scatter(reference[i], reference[j], marker="*", color="black", s=90, label="injected")
    ax.set(xlabel=labels[i], ylabel=labels[j])
axes[0].legend()
fig.tight_layout()
plt.show()

These plots show the bootstrap cloud and central quantiles, without filtering draws by their likelihood values. With only a few draws, they are diagnostic plots, not reliable confidence contours. Check ND residuals, FD cross-section agreement, density floors, and stability under more training, larger integration grids, and more starts. The parameter space contains degeneracies; an undertrained model can prefer a different octant.

## 7. Save the run recipe

The manifest records input hashes, package versions, seeds, and numerical settings. The checkpoint and fit archive can be inspected independently of this notebook. Outputs are ignored by Git so that re-running the tutorial keeps the source tree clean.

In [ ]:
manifest = dict(
    profile=PROFILE, device=str(rc.device), settings=cfg, seeds=seeds,
    python=platform.python_version(),
    versions={name: importlib.metadata.version(name)
              for name in ("torch", "numpy", "scipy", "pandas", "h5py", "matplotlib")},
    inputs={str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in required},
    oscillations=setup.osc_params, likelihood=ft.LIKELIHOOD, exposure=ft.EXPOSURE,
    model=str(Path(model_path).relative_to(ROOT)),
    samples=str(sample_dir.relative_to(ROOT)), fit=str(Path(fit_path).relative_to(ROOT)),
    negative_grid_fraction=negative_fraction,
)
manifest_path = OUT / "run.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
pd.DataFrame(summary).to_csv(OUT / "bootstrap_summary.csv", index=False)
print("Run recipe:", manifest_path.relative_to(ROOT))
print("Model:", Path(model_path).relative_to(ROOT))
print("Fits:", Path(fit_path).relative_to(ROOT))

## 8. Optional: flux uncertainties and detector effects

The nominal run above assumes known fluxes and infinite ND statistics. The paper also studies correlated flux uncertainties, finite ND statistics, and detector smearing. Those require retraining and repeated FD analyses. A small nominal run cannot estimate their contribution.

The retained `ml4nu.campaign.fwdfold_shot` driver performs this longer calculation. It fits flux nuisance coefficients with the cross-section model, transfers the ND constraint to the FD through the joint covariance, and samples the remaining FD uncertainty. The covariance files are bundled with the package. [docs/tutorials.md](docs/tutorials.md) explains the controls, result schema, and how to build an ensemble.

The next cell is **off by default**. Setting `RUN_SYSTEMATICS=True` starts one full-budget draw. It can take substantial time and memory. One draw teaches the interface; multiple independent draws are needed for total uncertainties.

In [ ]:
RUN_SYSTEMATICS = False
if RUN_SYSTEMATICS:
    import subprocess
    import sys

    env = dict(os.environ, NSTEP="100000", N_BOOTS="4096", PROGRESS="1",
               PROP="regress", FLUXRESID="shot", LIKELIHOOD="extended",
               OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
    if DEVICE == "cpu":
        env["CUDA_VISIBLE_DEVICES"] = ""
    subprocess.run([sys.executable, "-m", "ml4nu.campaign.fwdfold_shot", "0", "total", "ml"],
                   cwd=ROOT, env=env, check=True)
else:
    print("Optional systematics campaign skipped; the nominal tutorial is complete.")